In [ ]:
import sys
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import math
import os
import pickle

In [3]:
import pandas as pd
from itertools import combinations

WY_graphs = []
prefix = "../data/WY/560"
for i in range(1, 47, 2):
    
    base = prefix + f"{i:02d}"
    
    p = pd.read_csv(f"{base}/people.txt", sep="\t")
    gqp = pd.read_csv(f"{base}/gq_people.txt", sep="\t")
    
    edges = []
    for _, r in p.iterrows():
      pid = f"P:{r.sp_id}"
      edges.append((pid, f"H:{r.sp_hh_id}", "household"))
      if r.school_id != "X":
          edges.append((pid, f"S:{r.school_id}", "school"))
      if r.work_id != "X":
          edges.append((pid, f"W:{r.work_id}", "work"))
    
    for _, r in gqp.iterrows():
      edges.append((f"P:{r.sp_id}", f"G:{r.sp_gq_id}", "gq"))
    
    B = nx.Graph()
    for u, v, t in edges:
      B.add_node(u, kind="person")
      B.add_node(v, kind="place")
      B.add_edge(u, v, rel=t)
    
    # person-person projection
    G_p = nx.Graph()
    places = [n for n, d in B.nodes(data=True) if d["kind"] == "place"]
    for place in places:
      people = [n for n in B.neighbors(place) if B.nodes[n]["kind"] == "person"]
      rel = place.split(":")[0]
      for a, b in combinations(people, 2):
          if G_p.has_edge(a, b):
              G_p[a][b]["weight"] += 1
              G_p[a][b]["rels"].add(rel)
          else:
              G_p.add_edge(a, b, weight=1, rels={rel})
              
    largest_nodes = max(nx.connected_components(G_p), key=len)
    G_sub = G_p.subgraph(largest_nodes).copy()
    
    print("bipartite:", B.number_of_nodes(), B.number_of_edges())
    print("person-person:", G_p.number_of_nodes(), G_p.number_of_edges())
    print("subgraph person-person:", G_sub.number_of_nodes(), G_sub.number_of_edges())

    WY_graphs.append(G_sub)



with open("../data/WY_graphs.pkl", "wb") as f:
    pickle.dump(WY_graphs, f, protocol=pickle.HIGHEST_PROTOCOL)

bipartite: 54261 59946
person-person: 33961 4735491
subgraph person-person: 28901 2758799
bipartite: 17575 19738
person-person: 11245 363777
subgraph person-person: 9780 358231
bipartite: 65073 80685
person-person: 44496 9864724
subgraph person-person: 42497 9852037
bipartite: 23337 26868
person-person: 15148 996203
subgraph person-person: 13163 787592
bipartite: 21084 23636
person-person: 13124 718817
subgraph person-person: 11692 716873
bipartite: 10870 11776
person-person: 6656 396223
subgraph person-person: 5645 395167
bipartite: 57986 66306
person-person: 37683 3767941
subgraph person-person: 33318 3710343
bipartite: 19761 20956
person-person: 12190 818116
subgraph person-person: 9408 666115
bipartite: 7374 7596
person-person: 4306 112444
subgraph person-person: 3510 110711
bipartite: 13380 14468
person-person: 7919 322625
subgraph person-person: 6933 321380
bipartite: 134377 156902
person-person: 86942 14960479
subgraph person-person: 78901 14767349
bipartite: 26561 31052
person-

In [ ]:
with open("../data/WY_graphs.pkl", "rb") as f:
    WY_graphs = pickle.load(f)

In [ ]:
os.system("printf '\a'")

In [ ]:
for i in WY_graphs:
    print(nx.sigma(i))
    

In [ ]:
print(nx.sigma(WY_graphs[0]))